In [1]:
# ============================================================
# MODEL 3 — DATASET CONFIGURATION
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# ============================================================
# DEBUG SWITCH
# ============================================================

DEBUG = False

# True  -> small dataset for testing/debugging
# False -> full 200M-token dataset


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# TOKENIZER / MODEL SETTINGS
# ============================================================

VOCAB_SIZE = 32004
BLOCK_SIZE = 2048
TOKEN_DTYPE = np.uint16


# ============================================================
# TOKEN BUDGET
# ============================================================

FULL_TOTAL_TOKENS = 200_000_000

DEBUG_TOTAL_TOKENS = 2_000_000


TOTAL_TOKEN_BUDGET = (
    DEBUG_TOTAL_TOKENS
    if DEBUG
    else FULL_TOTAL_TOKENS
)


# ============================================================
# SOURCE RATIOS
# ============================================================

SOURCE_RATIOS = {
    "acts": 15,
    "high_courts": 35,
    "supreme_court": 25,
    "tribunals": 25,
}


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

TRAIN_RATIO = 0.98
TEST_RATIO = 0.02


# ============================================================
# DATA PATH
# ============================================================

TOKENIZED_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m"
)


SOURCE_FILES = {
    "acts":
        TOKENIZED_DIR / "acts_100m.bin",

    "high_courts":
        TOKENIZED_DIR / "high_courts_100m.bin",

    "supreme_court":
        TOKENIZED_DIR / "supreme_court_100m.bin",

    "tribunals":
        TOKENIZED_DIR / "tribunals_100m.bin",
}


# ============================================================
# OUTPUT
# ============================================================

RUN_NAME = (
    "debug"
    if DEBUG
    else "full"
)

OUTPUT_DIR = Path(
    "/kaggle/working/model3_data"
) / RUN_NAME

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DATALOADER
# ============================================================

NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
from torch.utils.data import DataLoader


BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1

TRAIN_NUM_WORKERS = 2
VAL_NUM_WORKERS = 2

print("=" * 70)
print("MODEL 3 DATASET CONFIG")
print("=" * 70)

print("DEBUG              :", DEBUG)
print("TOTAL TOKENS      :", f"{TOTAL_TOKEN_BUDGET:,}")
print("BLOCK SIZE        :", BLOCK_SIZE)
print("VOCAB SIZE        :", VOCAB_SIZE)
print("OUTPUT            :", OUTPUT_DIR)

MODEL 3 DATASET CONFIG
DEBUG              : False
TOTAL TOKENS      : 200,000,000
BLOCK SIZE        : 2048
VOCAB SIZE        : 32004
OUTPUT            : /kaggle/working/model3_data/full


In [2]:
# ============================================================
# VERIFY SOURCE FILES
# ============================================================

for source, path in SOURCE_FILES.items():

    print(
        f"{source:15s} -> {path}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Missing source file:\n{path}"
        )

print("\n✓ All source files found.")

acts            -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/acts_100m.bin
high_courts     -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/high_courts_100m.bin
supreme_court   -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/supreme_court_100m.bin
tribunals       -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/tribunals_100m.bin

✓ All source files found.


In [3]:
import shutil

source_dir = "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-3-finalmodels8k/pytorch/default/1"
destination_dir = "/kaggle/working"

# Merges the replica into the existing directory, overwriting matching files
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)


'/kaggle/working'

In [4]:
# ============================================================
# MEMORY-MAPPED TOKEN SOURCE
# ============================================================

class TokenFile:

    def __init__(
        self,
        path,
        dtype=np.uint16,
    ):

        self.path = Path(path)
        self.dtype = np.dtype(dtype)

        file_size = (
            self.path.stat().st_size
        )

        if (
            file_size
            % self.dtype.itemsize
            != 0
        ):

            raise ValueError(
                f"Invalid token file: {self.path}"
            )

        self.n_tokens = (
            file_size
            // self.dtype.itemsize
        )

        self.data = np.memmap(
            self.path,
            dtype=self.dtype,
            mode="r",
            shape=(self.n_tokens,),
        )


    def read(
        self,
        start,
        count,
    ):

        return np.asarray(
            self.data[
                start:start + count
            ]
        )

In [5]:
# ============================================================
# LOAD ALL FOUR SOURCES
# ============================================================

SOURCE_DATA = {}

for source, path in SOURCE_FILES.items():

    SOURCE_DATA[source] = TokenFile(
        path,
        dtype=TOKEN_DTYPE,
    )

    available = (
        SOURCE_DATA[source].n_tokens
    )

    print(
        f"{source:15s} "
        f"{available:,} tokens"
    )


print("\n✓ All sources loaded using mmap.")

acts            256,521,619 tokens
high_courts     242,356,017 tokens
supreme_court   241,501,699 tokens
tribunals       241,779,387 tokens

✓ All sources loaded using mmap.


In [6]:
# ============================================================
# SOURCE TOKEN BUDGET
# ============================================================

SOURCE_BUDGET = {}

for source, ratio in SOURCE_RATIOS.items():

    SOURCE_BUDGET[source] = int(
        TOTAL_TOKEN_BUDGET
        * ratio
        / 100
    )


print("=" * 70)
print("SOURCE BUDGET")
print("=" * 70)

for source in SOURCE_BUDGET:

    print(
        f"{source:15s}"
        f"{SOURCE_BUDGET[source]:>15,}"
    )


assert (
    sum(SOURCE_BUDGET.values())
    == TOTAL_TOKEN_BUDGET
)

SOURCE BUDGET
acts                30,000,000
high_courts         70,000,000
supreme_court       50,000,000
tribunals           50,000,000


In [7]:
# ============================================================
# 98 : 2 SPLIT
# ============================================================

TRAIN_BUDGET = {}
TEST_BUDGET = {}

for source in SOURCE_BUDGET:

    total = SOURCE_BUDGET[source]

    train_tokens = int(
        total * TRAIN_RATIO
    )

    test_tokens = (
        total - train_tokens
    )

    TRAIN_BUDGET[source] = (
        train_tokens
    )

    TEST_BUDGET[source] = (
        test_tokens
    )


TOTAL_TRAIN_TOKENS = sum(
    TRAIN_BUDGET.values()
)

TOTAL_TEST_TOKENS = sum(
    TEST_BUDGET.values()
)


print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

for source in SOURCE_BUDGET:

    print(
        f"{source:15s}"
        f"total={SOURCE_BUDGET[source]:>12,} "
        f"train={TRAIN_BUDGET[source]:>12,} "
        f"test={TEST_BUDGET[source]:>10,}"
    )


print("-" * 70)

print(
    "TRAIN:",
    f"{TOTAL_TRAIN_TOKENS:,}"
)

print(
    "TEST :",
    f"{TOTAL_TEST_TOKENS:,}"
)

TRAIN / TEST SPLIT
acts           total=  30,000,000 train=  29,400,000 test=   600,000
high_courts    total=  70,000,000 train=  68,600,000 test= 1,400,000
supreme_court  total=  50,000,000 train=  49,000,000 test= 1,000,000
tribunals      total=  50,000,000 train=  49,000,000 test= 1,000,000
----------------------------------------------------------------------
TRAIN: 196,000,000
TEST : 4,000,000


In [8]:
# ============================================================
# CREATE MERGED DATASETS
# ============================================================

TRAIN_BIN = (
    OUTPUT_DIR / "train.bin"
)

TEST_BIN = (
    OUTPUT_DIR / "test.bin"
)


def create_empty_bin(
    path,
    num_tokens,
):

    with open(path, "wb") as f:

        f.truncate(
            num_tokens
            * np.dtype(
                TOKEN_DTYPE
            ).itemsize
        )


create_empty_bin(
    TRAIN_BIN,
    TOTAL_TRAIN_TOKENS,
)

create_empty_bin(
    TEST_BIN,
    TOTAL_TEST_TOKENS,
)


train_mm = np.memmap(
    TRAIN_BIN,
    dtype=TOKEN_DTYPE,
    mode="r+",
    shape=(TOTAL_TRAIN_TOKENS,),
)

test_mm = np.memmap(
    TEST_BIN,
    dtype=TOKEN_DTYPE,
    mode="r+",
    shape=(TOTAL_TEST_TOKENS,),
)


train_offset = 0
test_offset = 0


# ============================================================
# COPY EACH SOURCE
# ============================================================

for source in SOURCE_RATIOS:

    source_data = SOURCE_DATA[
        source
    ]

    train_count = TRAIN_BUDGET[
        source
    ]

    test_count = TEST_BUDGET[
        source
    ]


    # --------------------------------------------------------
    # TRAIN PART
    # --------------------------------------------------------

    train_mm[
        train_offset:
        train_offset + train_count
    ] = source_data.read(
        0,
        train_count,
    )


    # --------------------------------------------------------
    # TEST PART
    # --------------------------------------------------------

    test_mm[
        test_offset:
        test_offset + test_count
    ] = source_data.read(
        train_count,
        test_count,
    )


    train_offset += train_count
    test_offset += test_count


    print(
        f"Processed {source}"
    )


train_mm.flush()
test_mm.flush()

del train_mm
del test_mm


print("\n✓ Merged dataset created.")

Processed acts
Processed high_courts
Processed supreme_court
Processed tribunals

✓ Merged dataset created.


In [9]:
# ============================================================
# CAUSAL LM DATASET
# ============================================================

class CausalLMDataset(
    Dataset
):

    def __init__(
        self,
        bin_path,
        num_tokens,
        block_size=2048,
    ):

        self.bin_path = Path(
            bin_path
        )

        self.block_size = (
            block_size
        )

        self.tokens = np.memmap(
            self.bin_path,
            dtype=TOKEN_DTYPE,
            mode="r",
            shape=(num_tokens,),
        )

        # +1 because target is shifted by 1
        self.num_examples = (
            (
                num_tokens
                - block_size
                - 1
            )
            // block_size
        )


    def __len__(self):

        return self.num_examples


    def __getitem__(
        self,
        idx,
    ):

        start = (
            idx
            * self.block_size
        )

        chunk = np.asarray(
            self.tokens[
                start:
                start
                + self.block_size
                + 1
            ],
            dtype=np.int64,
        )

        x = torch.from_numpy(
            chunk[:-1]
        )

        y = torch.from_numpy(
            chunk[1:]
        )

        return x, y

In [10]:
# ============================================================
# ARCHITECTURE 3
# E-LegalFormer
# Efficient Lightweight Transformer
# ============================================================

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class ELegalFormerConfig:

    vocab_size: int = 32004

    max_seq_len: int = 2048

    d_model: int = 768

    n_layers: int = 16

    n_q_heads: int = 12

    n_kv_heads: int = 4

    ffn_dim: int = 2048

    dropout: float = 0.0

    rope_theta: float = 10000.0


CONFIG = ELegalFormerConfig()


print(CONFIG)

ELegalFormerConfig(vocab_size=32004, max_seq_len=2048, d_model=768, n_layers=16, n_q_heads=12, n_kv_heads=4, ffn_dim=2048, dropout=0.0, rope_theta=10000.0)


In [11]:
# ============================================================
# MODEL 3 — TRAINING CONFIG
# ============================================================

EPOCHS = 1

LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1

WARMUP_RATIO = 0.03

GRAD_CLIP = 1.0

# Micro-batch on GPU
MICRO_BATCH_SIZE = 1

# Gradient accumulation
GRAD_ACCUM_STEPS = 8

# Logging / validation
LOG_EVERY = 50
VAL_EVERY = 500
SAVE_EVERY = 500

print("Configuration loaded.")
print("Micro batch:", MICRO_BATCH_SIZE)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print(
    "Effective batch size:",
    MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
)

Configuration loaded.
Micro batch: 1
Grad accumulation: 8
Effective batch size: 8


In [12]:
# ============================================================
# CONFIGURATION VALIDATION
# ============================================================

assert CONFIG.d_model % CONFIG.n_q_heads == 0

assert (
    CONFIG.n_q_heads
    % CONFIG.n_kv_heads
    == 0
)

HEAD_DIM = (
    CONFIG.d_model
    // CONFIG.n_q_heads
)

assert HEAD_DIM % 2 == 0


print(
    "Head dimension:",
    HEAD_DIM,
)

print(
    "Q heads:",
    CONFIG.n_q_heads,
)

print(
    "KV heads:",
    CONFIG.n_kv_heads,
)

print(
    "Q heads per KV head:",
    CONFIG.n_q_heads
    // CONFIG.n_kv_heads,
)

Head dimension: 64
Q heads: 12
KV heads: 4
Q heads per KV head: 3


In [13]:
# ============================================================
# DATASETS
# ============================================================

train_dataset = CausalLMDataset(
    TRAIN_BIN,
    TOTAL_TRAIN_TOKENS,
    BLOCK_SIZE,
)

validation_dataset = CausalLMDataset(
    TEST_BIN,
    TOTAL_TEST_TOKENS,
    BLOCK_SIZE,
)


# ============================================================
# DATALOADERS
# ============================================================

# ============================================================
# MODEL 3 — TRAINING DATALOADER
# ============================================================




train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    num_workers=TRAIN_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    persistent_workers=(
        TRAIN_NUM_WORKERS > 0
    ),
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=VAL_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(
        VAL_NUM_WORKERS > 0
    ),
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Effective batch size:",
    MICRO_BATCH_SIZE
    * GRAD_ACCUM_STEPS
)




Train batches: 95702
Validation batches: 1952
Effective batch size: 8


In [14]:
# ============================================================
# RMS NORMALIZATION
# ============================================================

class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-6,
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps


    def forward(self, x):

        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return (
            x * self.weight
        )

In [15]:
# ============================================================
# ROPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        dim,
        max_seq_len,
        theta=10000.0,
    ):

        super().__init__()

        inv_freq = (
            1.0
            / (
                theta
                ** (
                    torch.arange(
                        0,
                        dim,
                        2
                    ).float()
                    / dim
                )
            )
        )

        positions = torch.arange(
            max_seq_len
        )

        freqs = torch.outer(
            positions,
            inv_freq
        )

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

        self.max_seq_len = max_seq_len


    def forward(
        self,
        x,
        seq_len,
        start_pos=0,
    ):

        end_pos = start_pos + seq_len

        if end_pos > self.max_seq_len:
            raise ValueError(
                f"RoPE sequence exceeds max_seq_len: "
                f"start_pos={start_pos}, "
                f"seq_len={seq_len}, "
                f"max_seq_len={self.max_seq_len}"
            )

        cos = self.cos[
            start_pos:end_pos
        ]

        sin = self.sin[
            start_pos:end_pos
        ]

        return cos, sin


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]

    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        (-x2, x1),
        dim=-1,
    )


def apply_rope(
    q,
    k,
    cos,
    sin,
):

    # Convert [seq, dim/2] into
    # [seq, dim] by repeating.

    cos = torch.cat(
        [cos, cos],
        dim=-1,
    )

    sin = torch.cat(
        [sin, sin],
        dim=-1,
    )

    cos = cos.unsqueeze(0).unsqueeze(0)

    sin = sin.unsqueeze(0).unsqueeze(0)

    q = (
        q * cos
        + rotate_half(q) * sin
    )

    k = (
        k * cos
        + rotate_half(k) * sin
    )

    return q, k

In [16]:
# ============================================================
# GROUPED-QUERY ATTENTION
# ============================================================

class GroupedQueryAttention(
    nn.Module
):

    def __init__(
        self,
        config,
    ):

        super().__init__()

        self.d_model = (
            config.d_model
        )

        self.n_q_heads = (
            config.n_q_heads
        )

        self.n_kv_heads = (
            config.n_kv_heads
        )

        self.head_dim = (
            config.d_model
            // config.n_q_heads
        )

        self.q_proj = nn.Linear(
            config.d_model,
            config.n_q_heads
            * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            config.d_model,
            config.n_kv_heads
            * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            config.d_model,
            config.n_kv_heads
            * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            config.d_model,
            config.d_model,
            bias=False,
        )

        self.dropout = (
            config.dropout
        )


    def forward(
        self,
        x,
        cos,
        sin,
        past_key_value=None,
        use_cache=False,
    ):

        B, T, C = x.shape


        # ----------------------------------------------------
        # Q
        # ----------------------------------------------------

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_q_heads,
            self.head_dim,
        )

        q = q.transpose(
            1,
            2
        )
        # [B, n_q_heads, T, head_dim]


        # ----------------------------------------------------
        # K
        # ----------------------------------------------------

        k = self.k_proj(x)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(
            1,
            2
        )
        # [B, n_kv_heads, T, head_dim]


        # ----------------------------------------------------
        # V
        # ----------------------------------------------------

        v = self.v_proj(x)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.transpose(
            1,
            2
        )
        # [B, n_kv_heads, T, head_dim]


        # ----------------------------------------------------
        # RoPE
        # ----------------------------------------------------
        #
        # cos/sin are already selected for the correct
        # sequence positions by the model.
        #
        # For example:
        #
        # normal training:
        # positions = 0 ... T-1
        #
        # cached generation:
        # position = past_length
        #

        q, k = apply_rope(
            q,
            k,
            cos,
            sin,
        )


        # ----------------------------------------------------
        # KV CACHE
        # ----------------------------------------------------

        if past_key_value is not None:

            past_k, past_v = (
                past_key_value
            )

            # During incremental generation,
            # T should normally be 1.
            if T != 1:
                raise ValueError(
                    "When past_key_value is provided, "
                    "incremental decoding expects T=1."
                )

            k = torch.cat(
                [
                    past_k,
                    k,
                ],
                dim=2,
            )

            v = torch.cat(
                [
                    past_v,
                    v,
                ],
                dim=2,
            )


        # ----------------------------------------------------
        # GQA
        # ----------------------------------------------------

        repeat_factor = (
            self.n_q_heads
            // self.n_kv_heads
        )


        # ----------------------------------------------------
        # ATTENTION
        # ----------------------------------------------------
        #
        # No cache:
        #   normal causal self-attention
        #
        # Cache:
        #   q is only the new token and k/v contain
        #   all previous + current tokens.
        #
        #   Therefore we MUST NOT use is_causal=True
        #   here, because the cached keys are already
        #   valid past context.
        #

        if past_key_value is None:

            is_causal = True

        else:

            is_causal = False


        try:

            output = F.scaled_dot_product_attention(
                q,
                k,
                v,
                dropout_p=(
                    self.dropout
                    if self.training
                    else 0.0
                ),
                is_causal=is_causal,
                enable_gqa=True,
            )

        except TypeError:

            # ------------------------------------------------
            # Compatibility fallback
            # ------------------------------------------------

            k_for_attention = (
                k.repeat_interleave(
                    repeat_factor,
                    dim=1,
                )
            )

            v_for_attention = (
                v.repeat_interleave(
                    repeat_factor,
                    dim=1,
                )
            )

            output = F.scaled_dot_product_attention(
                q,
                k_for_attention,
                v_for_attention,
                dropout_p=(
                    self.dropout
                    if self.training
                    else 0.0
                ),
                is_causal=is_causal,
            )


        # ----------------------------------------------------
        # MERGE HEADS
        # ----------------------------------------------------

        output = output.transpose(
            1,
            2,
        )

        output = output.contiguous()

        output = output.view(
            B,
            T,
            C,
        )


        # ----------------------------------------------------
        # OUTPUT PROJECTION
        # ----------------------------------------------------

        output = self.o_proj(
            output
        )


        # ----------------------------------------------------
        # RETURN
        # ----------------------------------------------------

        if use_cache:

            return (
                output,
                (k, v),
            )

        return output

In [17]:
# ============================================================
# SWIGLU FEED-FORWARD NETWORK
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        d_model,
        ffn_dim,
    ):

        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            ffn_dim,
            d_model,
            bias=False,
        )


    def forward(self, x):

        gate = F.silu(
            self.gate_proj(x)
        )

        value = self.up_proj(x)

        return self.down_proj(
            gate * value
        )

In [18]:
# ============================================================
# EFFICIENT TRANSFORMER BLOCK
# ============================================================

class ELegalFormerBlock(
    nn.Module
):

    def __init__(
        self,
        config,
    ):

        super().__init__()


        self.attn_norm = RMSNorm(
            config.d_model
        )

        self.attention = (
            GroupedQueryAttention(
                config
            )
        )


        self.ffn_norm = RMSNorm(
            config.d_model
        )

        self.ffn = SwiGLU(
            config.d_model,
            config.ffn_dim,
        )


        self.dropout = nn.Dropout(
            config.dropout
        )


    def forward(
        self,
        x,
        cos,
        sin,
        past_key_value=None,
        use_cache=False,
    ):

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        h = self.attn_norm(x)


        if use_cache:

            h, present = self.attention(
                h,
                cos,
                sin,
                past_key_value=past_key_value,
                use_cache=True,
            )

        else:

            h = self.attention(
                h,
                cos,
                sin,
            )


        x = x + self.dropout(h)


        # ----------------------------------------------------
        # SwiGLU
        # ----------------------------------------------------

        h = self.ffn_norm(x)

        h = self.ffn(h)

        x = x + self.dropout(h)


        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------

        if use_cache:

            return (
                x,
                present,
            )

        return x

In [19]:
# ============================================================
# E-LEGALFORMER
# ============================================================

class ELegalFormer(
    nn.Module
):

    def __init__(
        self,
        config,
    ):

        super().__init__()

        self.config = config


        # ----------------------------------------------------
        # TOKEN EMBEDDING
        # ----------------------------------------------------

        self.token_embedding = (
            nn.Embedding(
                config.vocab_size,
                config.d_model,
            )
        )


        # ----------------------------------------------------
        # ROPE
        # ----------------------------------------------------

        self.rope = RotaryEmbedding(
            dim=(
                config.d_model
                // config.n_q_heads
            ),
            max_seq_len=(
                config.max_seq_len
            ),
            theta=config.rope_theta,
        )


        # ----------------------------------------------------
        # TRANSFORMER BLOCKS
        # ----------------------------------------------------

        self.layers = nn.ModuleList(
            [
                ELegalFormerBlock(
                    config
                )
                for _ in range(
                    config.n_layers
                )
            ]
        )


        # ----------------------------------------------------
        # FINAL NORMALIZATION
        # ----------------------------------------------------

        self.final_norm = RMSNorm(
            config.d_model
        )


        # ----------------------------------------------------
        # LM HEAD
        # ----------------------------------------------------

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )


        # Weight tying
        self.lm_head.weight = (
            self.token_embedding.weight
        )


        self.apply(
            self._init_weights
        )


    def _init_weights(
        self,
        module,
    ):

        if isinstance(
            module,
            nn.Linear,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )


        elif isinstance(
            module,
            nn.Embedding,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )


    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
    ):

        B, T = input_ids.shape


        # ----------------------------------------------------
        # Determine past sequence length
        # ----------------------------------------------------

        if past_key_values is None:

            past_len = 0

        else:

            if len(past_key_values) != len(
                self.layers
            ):
                raise ValueError(
                    "past_key_values must contain "
                    "one (K, V) pair per transformer layer."
                )

            first_past = (
                past_key_values[0]
            )

            if first_past is None:

                past_len = 0

            else:

                past_k, past_v = (
                    first_past
                )

                past_len = past_k.size(2)

                if past_k.size(2) != past_v.size(2):

                    raise ValueError(
                        "Past K and V must have "
                        "the same sequence length."
                    )


        # ----------------------------------------------------
        # Total sequence length
        # ----------------------------------------------------

        total_len = (
            past_len + T
        )


        if total_len > self.config.max_seq_len:

            raise ValueError(
                f"Total sequence length "
                f"{total_len} exceeds maximum "
                f"{self.config.max_seq_len}"
            )


        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        x = self.token_embedding(
            input_ids
        )


        # ----------------------------------------------------
        # RoPE frequencies
        # ----------------------------------------------------
        #
        # Normal training:
        #
        #   past_len = 0
        #   positions = 0 ... T-1
        #
        # Cached generation:
        #
        #   past_len = number of cached tokens
        #   positions = past_len ... past_len+T-1
        #

        cos, sin = self.rope(
            x,
            T,
            start_pos=past_len,
        )


        # Match device/dtype
        cos = cos.to(
            device=x.device,
            dtype=x.dtype,
        )

        sin = sin.to(
            device=x.device,
            dtype=x.dtype,
        )


        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        new_past_key_values = (
            [] if use_cache else None
        )


        for layer_idx, layer in enumerate(
            self.layers
        ):

            if past_key_values is None:

                layer_past = None

            else:

                layer_past = (
                    past_key_values[layer_idx]
                )


            if use_cache:

                x, present = layer(
                    x,
                    cos,
                    sin,
                    past_key_value=layer_past,
                    use_cache=True,
                )

                new_past_key_values.append(
                    present
                )

            else:

                x = layer(
                    x,
                    cos,
                    sin,
                )


        # ----------------------------------------------------
        # Final norm
        # ----------------------------------------------------

        x = self.final_norm(x)


        # ----------------------------------------------------
        # Language modeling head
        # ----------------------------------------------------

        logits = self.lm_head(x)


        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------

        if use_cache:

            return (
                logits,
                new_past_key_values,
            )

        return logits

In [20]:
# ============================================================
# CREATE MODEL 3
# ============================================================

model3 = ELegalFormer(
    CONFIG
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model3 = model3.to(device)


print(
    "Device:",
    device
)

Device: cpu


In [21]:
# ============================================================
# PARAMETER COUNT
# ============================================================

def count_parameters(
    model,
):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


num_params = count_parameters(
    model3
)


print("=" * 70)

print(
    "MODEL 3 PARAMETER COUNT"
)

print("=" * 70)

print(
    f"{num_params:,} parameters"
)

print(
    f"{num_params / 1e6:.2f} M parameters"
)


assert 50e6 <= num_params <= 300e6


print(
    "\n✓ Model satisfies the "
    "50M–300M requirement."
)

MODEL 3 PARAMETER COUNT
125,267,712 parameters
125.27 M parameters

✓ Model satisfies the 50M–300M requirement.


In [22]:
# ============================================================
# FORWARD PASS TEST
# ============================================================

model3.eval()

dummy_input = torch.randint(
    0,
    CONFIG.vocab_size,
    (
        2,
        CONFIG.max_seq_len,
    ),
    device=device,
)


with torch.no_grad():

    logits = model3(
        dummy_input
    )


print(
    "Input shape :",
    dummy_input.shape
)

print(
    "Logits shape:",
    logits.shape
)


assert logits.shape == (
    2,
    CONFIG.max_seq_len,
    CONFIG.vocab_size,
)


print(
    "\n✓ FORWARD PASS SUCCESSFUL"
)

Input shape : torch.Size([2, 2048])
Logits shape: torch.Size([2, 2048, 32004])

✓ FORWARD PASS SUCCESSFUL


In [23]:
# ============================================================
# MODEL 3 — TRAINING CONFIGURATION
# ============================================================

import os
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F


# ============================================================
# TRAINING SETTINGS
# ============================================================

EPOCHS = 1

LEARNING_RATE = 3e-4

MIN_LR = 3e-5

WEIGHT_DECAY = 0.1

WARMUP_RATIO = 0.03

GRAD_CLIP = 1.0


# ============================================================
# GRADIENT ACCUMULATION
# ============================================================

GRAD_ACCUM_STEPS = 8


# ============================================================
# LOGGING
# ============================================================

LOG_EVERY = 50

VAL_EVERY = 500

SAVE_EVERY = 100


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

CHECKPOINT_DIR = (
    Path("/kaggle/working")
    / "model3_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# MIXED PRECISION
# ============================================================

USE_AMP = (
    torch.cuda.is_available()
)

AMP_DTYPE = (
    torch.bfloat16
    if (
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    )
    else torch.float16
)


print("=" * 70)
print("TRAINING CONFIG")
print("=" * 70)

print("Epochs           :", EPOCHS)
print("Learning rate    :", LEARNING_RATE)
print("Minimum LR       :", MIN_LR)
print("Weight decay     :", WEIGHT_DECAY)
print("Warmup ratio     :", WARMUP_RATIO)
print("Grad clip        :", GRAD_CLIP)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("AMP              :", USE_AMP)
print("AMP dtype        :", AMP_DTYPE)
print("Checkpoint dir   :", CHECKPOINT_DIR)

TRAINING CONFIG
Epochs           : 1
Learning rate    : 0.0003
Minimum LR       : 3e-05
Weight decay     : 0.1
Warmup ratio     : 0.03
Grad clip        : 1.0
Grad accumulation: 8
AMP              : False
AMP dtype        : torch.float16
Checkpoint dir   : /kaggle/working/model3_checkpoints


In [24]:
# ============================================================
# OPTIMIZER PARAMETER GROUPS
# ============================================================

decay_params = []
no_decay_params = []


for name, param in model3.named_parameters():

    if not param.requires_grad:
        continue

    if (
        param.ndim >= 2
        and "norm" not in name.lower()
    ):

        decay_params.append(param)

    else:

        no_decay_params.append(param)


print(
    "Decay parameters:",
    sum(
        p.numel()
        for p in decay_params
    )
)

print(
    "No-decay parameters:",
    sum(
        p.numel()
        for p in no_decay_params
    )
)


optimizer = torch.optim.AdamW(

    [
        {
            "params": decay_params,
            "weight_decay": WEIGHT_DECAY,
        },

        {
            "params": no_decay_params,
            "weight_decay": 0.0,
        },
    ],

    lr=LEARNING_RATE,

    betas=(0.9, 0.95),

    eps=1e-8,
)

Decay parameters: 125242368
No-decay parameters: 25344


In [25]:
# ============================================================
# TRAINING STEP CALCULATION
# ============================================================

STEPS_PER_EPOCH = (
    len(train_loader)
    // GRAD_ACCUM_STEPS
)

TOTAL_TRAIN_STEPS = (
    STEPS_PER_EPOCH
    * EPOCHS
)

WARMUP_STEPS = int(
    TOTAL_TRAIN_STEPS
    * WARMUP_RATIO
)


print("=" * 70)
print("TRAINING STEPS")
print("=" * 70)

print(
    "Micro batches / epoch:",
    len(train_loader)
)

print(
    "Optimizer steps / epoch:",
    STEPS_PER_EPOCH
)

print(
    "Total optimizer steps:",
    TOTAL_TRAIN_STEPS
)

print(
    "Warmup steps:",
    WARMUP_STEPS
)

TRAINING STEPS
Micro batches / epoch: 95702
Optimizer steps / epoch: 11962
Total optimizer steps: 11962
Warmup steps: 358


In [26]:
# ============================================================
# LEARNING RATE SCHEDULER
# ============================================================

def get_lr(
    step,
):

    # --------------------------------------------------------
    # Warmup
    # --------------------------------------------------------

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / max(
                1,
                WARMUP_STEPS
            )
        )


    # --------------------------------------------------------
    # Cosine decay
    # --------------------------------------------------------

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TOTAL_TRAIN_STEPS
        - WARMUP_STEPS
    )

    progress = min(
        1.0,
        max(
            0.0,
            progress
        )
    )


    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )


    return (
        MIN_LR
        + (
            LEARNING_RATE
            - MIN_LR
        )
        * cosine
    )


def set_learning_rate(
    optimizer,
    lr,
):

    for group in optimizer.param_groups:

        group["lr"] = lr

In [27]:
# ============================================================
# AUTOMATIC MIXED PRECISION
# ============================================================

if (
    USE_AMP
    and AMP_DTYPE == torch.float16
):

    scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    scaler = None


print(
    "GradScaler:",
    scaler is not None
)

GradScaler: False


In [28]:
# ============================================================
# LANGUAGE MODEL LOSS
# ============================================================

def causal_lm_loss(
    logits,
    labels,
):

    B, T, V = logits.shape


    loss = F.cross_entropy(

        logits.reshape(
            B * T,
            V
        ),

        labels.reshape(
            B * T
        ),
    )


    return loss

In [29]:
# ============================================================
# SINGLE TRAINING STEP
# ============================================================

def train_step(
    model,
    input_ids,
    labels,
):

    input_ids = input_ids.to(
        device,
        non_blocking=True
    )

    labels = labels.to(
        device,
        non_blocking=True
    )


    if USE_AMP:

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):

            logits = model(
                input_ids
            )

            loss = causal_lm_loss(
                logits,
                labels
            )

    else:

        logits = model(
            input_ids
        )

        loss = causal_lm_loss(
            logits,
            labels
        )


    return loss

In [30]:
# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    max_batches=None,
):

    model.eval()


    total_loss = 0.0
    total_batches = 0


    for batch_idx, (
        input_ids,
        labels,
    ) in enumerate(loader):


        if (
            max_batches is not None
            and batch_idx >= max_batches
        ):

            break


        input_ids = input_ids.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        if USE_AMP:

            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
            ):

                logits = model(
                    input_ids
                )

                loss = causal_lm_loss(
                    logits,
                    labels
                )

        else:

            logits = model(
                input_ids
            )

            loss = causal_lm_loss(
                logits,
                labels
            )


        total_loss += (
            loss.item()
        )

        total_batches += 1


    if total_batches == 0:

        return float("inf")


    return (
        total_loss
        / total_batches
    )

In [31]:
# ============================================================
# PERPLEXITY
# ============================================================

def perplexity(
    loss,
):

    try:

        return math.exp(loss)

    except OverflowError:

        return float("inf")

In [32]:
# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(
    path,
    model,
    optimizer,
    scaler,
    step,
    best_val_loss,
):

    checkpoint = {

        "step": step,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_val_loss":
            best_val_loss,

        "config":
            vars(CONFIG),

    }


    if scaler is not None:

        checkpoint[
            "scaler_state_dict"
        ] = scaler.state_dict()


    torch.save(
        checkpoint,
        path
    )


    print(
        f"Checkpoint saved: {path}"
    )

In [33]:
# ============================================================
# LOAD CHECKPOINT
# ============================================================

def load_checkpoint(
    path,
    model,
    optimizer,
    scaler=None,
):

    checkpoint = torch.load(
        path,
        map_location=device,
    )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )


    if (
        scaler is not None
        and "scaler_state_dict"
        in checkpoint
    ):

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )


    step = checkpoint.get(
        "step",
        0,
    )


    best_val_loss = checkpoint.get(
        "best_val_loss",
        float("inf"),
    )


    print(
        "Checkpoint loaded:"
    )

    print(
        "Step:",
        step
    )

    print(
        "Best validation loss:",
        best_val_loss
    )


    return (
        step,
        best_val_loss,
    )

In [34]:
def cleanup_old_checkpoints(
    checkpoint_dir,
    keep=2,
):

    files = sorted(
        Path(checkpoint_dir).glob(
            "checkpoint_step_*.pt"
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )


    for old_file in files[keep:]:

        old_file.unlink(
            missing_ok=True
        )

        print(
            "Deleted old checkpoint:",
            old_file.name
        )

In [35]:
# ============================================================
# TRAINING HISTORY
# ============================================================

history = {

    "step": [],

    "train_loss": [],

    "val_loss": [],

    "train_ppl": [],

    "val_ppl": [],

    "learning_rate": [],

}

In [36]:
# ============================================================
# MODEL 3 — AUTO RESUME / START FROM SCRATCH
# ============================================================

import re
from pathlib import Path
import torch


# ------------------------------------------------------------
# CHECKPOINT DIRECTORY
# ------------------------------------------------------------

CHECKPOINT_DIR = Path(
    "/kaggle/working/"
    "model3_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# FIND NUMBERED CHECKPOINTS
# Example:
# checkpoint_step_100.pt
# checkpoint_step_1100.pt
# checkpoint_step_1300.pt
# ------------------------------------------------------------

checkpoint_files = []

pattern = re.compile(
    r"checkpoint_step_(\d+)\.pt$"
)


for path in CHECKPOINT_DIR.glob(
    "checkpoint_step_*.pt"
):

    match = pattern.match(
        path.name
    )

    if match:

        step = int(
            match.group(1)
        )

        checkpoint_files.append(
            (step, path)
        )


# Sort newest first
checkpoint_files.sort(
    key=lambda x: x[0],
    reverse=True
)


# ------------------------------------------------------------
# DEFAULT: START FROM SCRATCH
# ------------------------------------------------------------

global_step = 0

best_val_loss = float("inf")

resume_checkpoint = None


# ------------------------------------------------------------
# TRY LATEST VALID CHECKPOINT
# ------------------------------------------------------------

for step, path in checkpoint_files:

    try:

        print(
            f"Trying checkpoint: "
            f"{path.name}"
        )

        checkpoint = torch.load(
            path,
            map_location=device
        )

        # -----------------------------------------------
        # MODEL
        # -----------------------------------------------

        model3.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )


        # -----------------------------------------------
        # OPTIMIZER
        # -----------------------------------------------

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )


        # -----------------------------------------------
        # AMP SCALER
        # -----------------------------------------------

        if (
            scaler is not None
            and "scaler_state_dict"
            in checkpoint
        ):

            scaler.load_state_dict(
                checkpoint[
                    "scaler_state_dict"
                ]
            )


        # -----------------------------------------------
        # TRAINING STATE
        # -----------------------------------------------

        global_step = checkpoint.get(
            "step",
            step
        )

        best_val_loss = checkpoint.get(
            "best_val_loss",
            float("inf")
        )


        resume_checkpoint = path

        print(
            f"\n✓ Resuming from:"
            f" {path.name}"
        )

        break


    except Exception as e:

        print(
            f"⚠ Could not load "
            f"{path.name}"
        )

        print(
            f"  Reason: {e}"
        )

        print(
            "  Trying previous checkpoint..."
        )


# ------------------------------------------------------------
# NO VALID NUMBERED CHECKPOINT
# ------------------------------------------------------------

if resume_checkpoint is None:

    print(
        "\nNo valid numbered checkpoint found."
    )

    print(
        "Starting training from scratch."
    )

    global_step = 0

    best_val_loss = float("inf")


# ------------------------------------------------------------
# CALCULATE RESUME POSITION
# ------------------------------------------------------------

SKIP_BATCHES = (
    global_step
    * GRAD_ACCUM_STEPS
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING STATE")
print("=" * 70)

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR
)

print(
    "Resume checkpoint:",
    (
        resume_checkpoint
        if resume_checkpoint is not None
        else "NONE"
    )
)

print(
    "Global step:",
    global_step
)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Micro-batches already processed:",
    f"{SKIP_BATCHES:,}"
)

print(
    "Total optimizer steps:",
    TOTAL_TRAIN_STEPS
)


if global_step >= TOTAL_TRAIN_STEPS:

    print(
        "\n✓ Training is already complete."
    )

else:

    print(
        "\n✓ Ready to continue training."
    )

Trying checkpoint: checkpoint_step_8000.pt

✓ Resuming from: checkpoint_step_8000.pt

TRAINING STATE
Checkpoint directory: /kaggle/working/model3_checkpoints
Resume checkpoint: /kaggle/working/model3_checkpoints/checkpoint_step_8000.pt
Global step: 8000
Best validation loss: 1.2227919439558077
Micro-batches already processed: 64,000
Total optimizer steps: 11962

✓ Ready to continue training.


In [37]:
from tqdm.auto import tqdm
import time

# ============================================================
# MODEL 3 — RESUMABLE TRAINING LOOP
# TRAIN ONLY UP TO MAX_TRAIN_STEPS OPTIMIZER STEPS
# ============================================================

MAX_TRAIN_STEPS = 8000


# ------------------------------------------------------------
# BASIC CHECK
# ------------------------------------------------------------

if global_step > MAX_TRAIN_STEPS:

    raise ValueError(
        f"Current checkpoint step ({global_step}) "
        f"is already greater than MAX_TRAIN_STEPS "
        f"({MAX_TRAIN_STEPS})."
    )


# ------------------------------------------------------------
# TRAINING SETUP
# ------------------------------------------------------------

model3.train()

optimizer.zero_grad(
    set_to_none=True
)

start_time = time.time()


print("=" * 70)
print("MODEL 3 TRAINING")
print("=" * 70)

print(
    f"Starting optimizer step : {global_step}"
)

print(
    f"Target optimizer step   : {MAX_TRAIN_STEPS}"
)

print(
    f"Remaining steps         : "
    f"{MAX_TRAIN_STEPS - global_step}"
)

print(
    f"Best validation loss    : "
    f"{best_val_loss:.6f}"
)

print(
    f"Skipping micro-batches  : "
    f"{SKIP_BATCHES:,}"
)


# ============================================================
# ALREADY COMPLETE?
# ============================================================

if global_step == MAX_TRAIN_STEPS:

    print()
    print(
        f"✓ Model is already at step "
        f"{MAX_TRAIN_STEPS}."
    )

else:

    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(EPOCHS):

        # ----------------------------------------------------
        # If target already reached, stop
        # ----------------------------------------------------

        if global_step >= MAX_TRAIN_STEPS:
            break


        print()
        print("=" * 70)
        print(
            f"EPOCH {epoch + 1}/{EPOCHS}"
        )
        print("=" * 70)


        accumulated_loss = 0.0


        # ----------------------------------------------------
        # PROGRESS BAR
        # ----------------------------------------------------

        pbar = tqdm(
            total=len(train_loader),

            initial=min(
                SKIP_BATCHES,
                len(train_loader)
            ),

            desc=(
                f"Epoch {epoch + 1}/{EPOCHS}"
            ),

            unit="batch",

            dynamic_ncols=True,
        )


        # ====================================================
        # BATCH LOOP
        # ====================================================

        for batch_idx, (
            input_ids,
            labels
        ) in enumerate(train_loader):


            # ------------------------------------------------
            # STOP EXACTLY AT MAX_TRAIN_STEPS
            # ------------------------------------------------

            if global_step >= MAX_TRAIN_STEPS:

                break


            # ------------------------------------------------
            # SKIP BATCHES ALREADY PROCESSED
            # ------------------------------------------------

            if batch_idx < SKIP_BATCHES:

                pbar.update(1)

                continue


            # ------------------------------------------------
            # FORWARD + LOSS
            # ------------------------------------------------

            loss = train_step(
                model3,
                input_ids,
                labels,
            )


            # ------------------------------------------------
            # GRADIENT ACCUMULATION
            # ------------------------------------------------

            loss_for_backward = (
                loss / GRAD_ACCUM_STEPS
            )


            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            if scaler is not None:

                scaler.scale(
                    loss_for_backward
                ).backward()

            else:

                loss_for_backward.backward()


            accumulated_loss += (
                loss.item()
            )


            # ------------------------------------------------
            # OPTIMIZER UPDATE
            # ------------------------------------------------

            if (
                (batch_idx + 1)
                % GRAD_ACCUM_STEPS
                == 0
            ):


                # --------------------------------------------
                # GRADIENT CLIPPING
                # --------------------------------------------

                if scaler is not None:

                    scaler.unscale_(
                        optimizer
                    )


                torch.nn.utils.clip_grad_norm_(
                    model3.parameters(),
                    GRAD_CLIP,
                )


                # --------------------------------------------
                # LEARNING RATE
                # --------------------------------------------

                lr = get_lr(
                    global_step
                )

                set_learning_rate(
                    optimizer,
                    lr,
                )


                # --------------------------------------------
                # OPTIMIZER STEP
                # --------------------------------------------

                if scaler is not None:

                    scaler.step(
                        optimizer
                    )

                    scaler.update()

                else:

                    optimizer.step()


                optimizer.zero_grad(
                    set_to_none=True
                )


                # --------------------------------------------
                # UPDATE GLOBAL STEP
                # --------------------------------------------

                global_step += 1


                # --------------------------------------------
                # AVERAGE LOSS
                # --------------------------------------------

                avg_loss = (
                    accumulated_loss
                    / GRAD_ACCUM_STEPS
                )

                accumulated_loss = 0.0


                # =================================================
                # LOGGING
                # =================================================

                if (
                    global_step
                    % LOG_EVERY
                    == 0
                ):

                    elapsed_minutes = (
                        time.time()
                        - start_time
                    ) / 60.0


                    print(
                        f"Step {global_step:6d} | "
                        f"Loss {avg_loss:.4f} | "
                        f"LR {lr:.2e} | "
                        f"Time "
                        f"{elapsed_minutes:.1f} min"
                    )


                # =================================================
                # VALIDATION
                # =================================================

                if (
                    global_step
                    % VAL_EVERY
                    == 0
                ):

                    print()

                    print(
                        f"Running validation "
                        f"at step {global_step}..."
                    )


                    val_loss = evaluate_model(
                        model3,
                        validation_loader,
                    )


                    val_ppl = perplexity(
                        val_loss
                    )


                    print(
                        f"Validation | "
                        f"step={global_step} | "
                        f"loss={val_loss:.4f} | "
                        f"ppl={val_ppl:.2f}"
                    )


                    # FIX: record history for the training-curve plot
                    history["step"].append(global_step)
                    history["train_loss"].append(avg_loss)
                    history["val_loss"].append(val_loss)
                    history["train_ppl"].append(perplexity(avg_loss))
                    history["val_ppl"].append(val_ppl)
                    history["learning_rate"].append(lr)


                    # ----------------------------------------
                    # BEST MODEL
                    # ----------------------------------------

                    if (
                        val_loss
                        < best_val_loss
                    ):

                        best_val_loss = (
                            val_loss
                        )


                        print(
                            "✓ New best model"
                        )


                        save_checkpoint(
                            CHECKPOINT_DIR
                            / "best_model.pt",

                            model3,

                            optimizer,

                            scaler,

                            global_step,

                            best_val_loss,
                        )


                    model3.train()


                # =================================================
                # PERIODIC CHECKPOINT
                # =================================================

                if (
                    global_step
                    % SAVE_EVERY
                    == 0
                ):

                    checkpoint_path = (
                        CHECKPOINT_DIR
                        / (
                            f"checkpoint_step_"
                            f"{global_step}.pt"
                        )
                    )


                    save_checkpoint(
                        checkpoint_path,

                        model3,

                        optimizer,

                        scaler,

                        global_step,

                        best_val_loss,
                    )


                    cleanup_old_checkpoints(
                        CHECKPOINT_DIR,
                        keep=3,
                    )


                # =================================================
                # FINAL TARGET CHECK
                # =================================================

                if (
                    global_step
                    >= MAX_TRAIN_STEPS
                ):

                    print()
                    print(
                        "=" * 70
                    )

                    print(
                        f"✓ Reached target "
                        f"step {MAX_TRAIN_STEPS}"
                    )

                    print(
                        "Stopping training."
                    )

                    print(
                        "=" * 70
                    )

                    break


            # ----------------------------------------------------
            # UPDATE PROGRESS BAR
            # ----------------------------------------------------

            pbar.update(1)


            # ----------------------------------------------------
            # CURRENT STATUS
            # ----------------------------------------------------

            pbar.set_postfix(
                step=global_step,

                loss=f"{loss.item():.4f}",

                lr=(
                    f"{optimizer.param_groups[0]['lr']:.2e}"
                ),
            )


        pbar.close()


        # --------------------------------------------------------
        # STOP AFTER TARGET STEP
        # --------------------------------------------------------

        if (
            global_step
            >= MAX_TRAIN_STEPS
        ):

            break


# ============================================================
# FINAL SAFETY CHECKPOINT
# ============================================================

if (
    global_step
    == MAX_TRAIN_STEPS
):

    final_checkpoint = (
        CHECKPOINT_DIR
        / f"checkpoint_step_"
        f"{MAX_TRAIN_STEPS}.pt"
    )


    # Save only if it does not already exist
    if not final_checkpoint.exists():

        save_checkpoint(
            final_checkpoint,

            model3,

            optimizer,

            scaler,

            global_step,

            best_val_loss,
        )


# ============================================================
# FINAL STATUS
# ============================================================

print()
print("=" * 70)
print("TRAINING STOPPED")
print("=" * 70)

print(
    f"Final optimizer step : "
    f"{global_step}"
)

print(
    f"Target optimizer step : "
    f"{MAX_TRAIN_STEPS}"
)

print(
    f"Best validation loss : "
    f"{best_val_loss:.6f}"
)

print(
    f"Best validation PPL  : "
    f"{perplexity(best_val_loss):.2f}"
)

MODEL 3 TRAINING
Starting optimizer step : 8000
Target optimizer step   : 8000
Remaining steps         : 0
Best validation loss    : 1.222792
Skipping micro-batches  : 64,000

✓ Model is already at step 8000.

TRAINING STOPPED
Final optimizer step : 8000
Target optimizer step : 8000
Best validation loss : 1.222792
Best validation PPL  : 3.40


In [38]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_PATH = (
    CHECKPOINT_DIR
    / "model3_final.pt"
)


torch.save(
    {
        "model_state_dict":
            model3.state_dict(),

        "config":
            vars(CONFIG),

        "parameter_count":
            count_parameters(model3),

        "best_val_loss":
            best_val_loss,

        "best_val_perplexity":
            perplexity(
                best_val_loss
            ),
    },

    FINAL_MODEL_PATH,
)


print(
    "Final model saved to:"
)

print(
    FINAL_MODEL_PATH
)

Final model saved to:
/kaggle/working/model3_checkpoints/model3_final.pt


In [39]:
# ============================================================
# TRAINING CURVES
# ============================================================

import matplotlib.pyplot as plt


if len(history["step"]) > 0:

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        history["step"],
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        history["step"],
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel(
        "Training Step"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        "Model 3 Training / Validation Loss"
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.show()

In [40]:
# ============================================================
# LOAD TRAINED BPE TOKENIZER
# ============================================================

import sys
from pathlib import Path


# ------------------------------------------------------------
# PATH OF TOKENIZER CREATED IN TOKENIZER NOTEBOOK
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer"
)


if not TOKENIZER_DIR.exists():

    raise FileNotFoundError(
        f"Tokenizer directory not found:\n"
        f"{TOKENIZER_DIR}\n\n"
        "Make sure the tokenizer notebook has been "
        "run and its final_tokenizer directory is "
        "available in this notebook."
    )


TOKENIZER_PY = (
    TOKENIZER_DIR / "tokenizer.py"
)


if not TOKENIZER_PY.exists():

    raise FileNotFoundError(
        f"tokenizer.py not found:\n"
        f"{TOKENIZER_PY}"
    )


# ------------------------------------------------------------
# IMPORT tokenizer.py
# ------------------------------------------------------------

sys.path.insert(
    0,
    str(TOKENIZER_DIR)
)

from tokenizer import load_tokenizer


# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer, tokenizer_config = (
    load_tokenizer(
        TOKENIZER_DIR
    )
)


print("=" * 70)
print("TOKENIZER LOADED")
print("=" * 70)

print(
    "Tokenizer directory:",
    TOKENIZER_DIR
)

print(
    "Vocabulary size:",
    tokenizer.get_vocab_size()
)

print(
    "PAD ID:",
    tokenizer.pad_id
)

print(
    "UNK ID:",
    tokenizer.unk_id
)

print(
    "BOS ID:",
    tokenizer.bos_id
)

print(
    "EOS ID:",
    tokenizer.eos_id
)

TOKENIZER LOADED
Tokenizer directory: /kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer
Vocabulary size: 32004
PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3


In [41]:
# ============================================================
# MODEL 3 GENERATION — KV CACHE  (FIXED)
# ============================================================
#
# Fixes vs. previous version:
#   1. Special tokens (PAD/UNK/BOS/EOS) can no longer be sampled.
#      They decode to nothing and waste the token budget.
#   2. stop_words: generation stops as soon as the continuation
#      contains more than `stop_words` words (~40% faster).
#   3. Optional repetition_penalty (1.0 = off).
#   4. The caller is responsible for leaving room in the context
#      window (prompt_len + max_new_tokens <= max_seq_len).
# ============================================================

SPECIAL_TOKEN_IDS = sorted({
    tokenizer.pad_id,
    tokenizer.unk_id,
    tokenizer.bos_id,
    tokenizer.eos_id,
})

STOP_CHECK_EVERY = 32


@torch.inference_mode()
def generate(
    model,
    input_ids,
    max_new_tokens=128,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    stop_words=None,
    repetition_penalty=1.0,
):

    model.eval()

    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    if top_k is not None and top_k <= 0:
        raise ValueError("top_k must be > 0 or None")
    if top_p is not None and not (0.0 < top_p <= 1.0):
        raise ValueError("top_p must be in (0, 1]")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be >= 0")
    if max_new_tokens == 0:
        return input_ids

    # Keep room for generation inside the context window
    max_prompt = CONFIG.max_seq_len - max_new_tokens
    if max_prompt < 1:
        raise ValueError("max_new_tokens leaves no room for the prompt")
    input_ids = input_ids[:, -max_prompt:]

    prompt_len = input_ids.size(1)

    # --------------------------------------------------------
    # SAMPLING HELPER
    # --------------------------------------------------------

    def sample_next_token(logits, generated):

        logits = logits.float()

        # Repetition penalty on tokens already generated
        if repetition_penalty != 1.0 and generated.numel() > 0:
            prev = torch.unique(generated)
            vals = logits[:, prev]
            vals = torch.where(
                vals > 0,
                vals / repetition_penalty,
                vals * repetition_penalty,
            )
            logits[:, prev] = vals

        logits = logits / temperature

        # Never sample special tokens
        logits[:, SPECIAL_TOKEN_IDS] = float("-inf")

        logits = torch.nan_to_num(
            logits, nan=-1e4, posinf=1e4, neginf=float("-inf")
        )

        # Top-k
        if top_k is not None:
            k = min(top_k, logits.size(-1))
            values, _ = torch.topk(logits, k)
            min_value = values[:, -1].unsqueeze(-1)
            logits = logits.masked_fill(logits < min_value, float("-inf"))

        # Top-p
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cumulative > top_p
            remove[:, 1:] = remove[:, :-1].clone()
            remove[:, 0] = False
            sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
            logits = torch.full_like(logits, float("-inf"))
            logits.scatter_(1, sorted_indices, sorted_logits)

        probs = torch.softmax(logits, dim=-1)
        probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)
        prob_sum = probs.sum(dim=-1, keepdim=True)

        if torch.any(prob_sum <= 0):
            return torch.argmax(logits, dim=-1, keepdim=True)

        return torch.multinomial(probs / prob_sum, num_samples=1)

    def forward(ids, past=None):
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                return model(ids, past_key_values=past, use_cache=True)
        return model(ids, past_key_values=past, use_cache=True)

    # --------------------------------------------------------
    # PREFILL
    # --------------------------------------------------------

    logits, past_key_values = forward(input_ids)
    next_token = sample_next_token(
        logits[:, -1, :], input_ids[0, prompt_len:]
    )
    input_ids = torch.cat([input_ids, next_token], dim=1)

    # --------------------------------------------------------
    # INCREMENTAL DECODING
    # --------------------------------------------------------

    for _ in range(max_new_tokens - 1):

        if input_ids.size(1) >= CONFIG.max_seq_len:
            print("    ⚠ context window full — stopping early")
            break

        # Early stop once we have more than stop_words words
        n_new = input_ids.size(1) - prompt_len
        if stop_words is not None and n_new % STOP_CHECK_EVERY == 0:
            text = tokenizer.decode(input_ids[0, prompt_len:].tolist())
            if len(text.split()) > stop_words:
                break

        logits, past_key_values = forward(next_token, past_key_values)
        next_token = sample_next_token(
            logits[:, -1, :], input_ids[0, prompt_len:]
        )
        input_ids = torch.cat([input_ids, next_token], dim=1)

    return input_ids


In [42]:
# ============================================================
# LOAD TRAINED MODEL 3
# ============================================================

BEST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "best_model.pt"
)


if not BEST_CHECKPOINT.exists():

    raise FileNotFoundError(
        f"Best Model 3 checkpoint not found:\n"
        f"{BEST_CHECKPOINT}"
    )


checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
)


model3.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


model3 = model3.to(device)

model3.eval()


print("=" * 70)
print("MODEL 3 CHECKPOINT LOADED")
print("=" * 70)

print(
    "Checkpoint:",
    BEST_CHECKPOINT
)

print(
    "Training step:",
    checkpoint.get(
        "step",
        "unknown"
    )
)

print(
    "Best validation loss:",
    checkpoint.get(
        "best_val_loss",
        "unknown"
    )
)


MODEL 3 CHECKPOINT LOADED
Checkpoint: /kaggle/working/model3_checkpoints/best_model.pt
Training step: 8000
Best validation loss: 1.2227919439558077


In [43]:
# ============================================================
# SINGLE PROMPT TEST GENERATION
# ============================================================

import re


# ------------------------------------------------------------
# WORD UTILITIES
# ------------------------------------------------------------

def count_words(text):
    return len(
        re.findall(
            r"\S+",
            text
        )
    )


def first_n_words(
    text,
    n=128,
):
    words = re.findall(
        r"\S+",
        text
    )

    return " ".join(
        words[:n]
    )


# ============================================================
# TEST PROMPT
# ============================================================

prompt = """
The court considered whether the petitioner was entitled
to relief under the provisions of the Act and whether the
order passed by the lower court was legally sustainable.
"""


# ============================================================
# TOKENIZE PROMPT
# ============================================================

prompt_ids = tokenizer.encode(
    prompt
)


print(
    "Prompt word count:",
    count_words(prompt)
)

print(
    "Prompt token count:",
    len(prompt_ids)
)


# ============================================================
# SAFETY CHECK
# ============================================================

if len(prompt_ids) > CONFIG.max_seq_len:

    print(
        "Warning: prompt exceeds model context."
    )

    prompt_ids = prompt_ids[
        -CONFIG.max_seq_len:
    ]


# ============================================================
# CONVERT TO TENSOR
# ============================================================

input_ids = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device,
)


# ============================================================
# GENERATE
#
# 128 words generally require more than 128 BPE tokens,
# so generate up to 512 new tokens.
# ============================================================

generated_ids = generate(
    model=model3,
    input_ids=input_ids,
    max_new_tokens=512,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    stop_words=128,
)


# ============================================================
# EXTRACT ONLY GENERATED TOKENS
# ============================================================

# generate() may left-truncate the prompt, so slice by the
# actual prompt length used rather than len(prompt_ids)
used_prompt_len = min(
    len(prompt_ids),
    CONFIG.max_seq_len - 512,
)

new_token_ids = generated_ids[
    0,
    used_prompt_len:
].tolist()


# ============================================================
# DECODE ONLY THE CONTINUATION
# ============================================================

continuation = tokenizer.decode(
    new_token_ids
)


# ============================================================
# KEEP EXACTLY 128 WORDS
# ============================================================

continuation = first_n_words(
    continuation,
    128
)


# ============================================================
# VALIDATE
# ============================================================

continuation_word_count = (
    count_words(
        continuation
    )
)


# ============================================================
# DISPLAY
# ============================================================

print()
print("=" * 70)
print("GENERATED CONTINUATION ONLY")
print("=" * 70)

print(continuation)

print()
print("=" * 70)
print("GENERATION STATS")
print("=" * 70)

print(
    "Generated token count:",
    len(new_token_ids)
)

print(
    "Final word count:",
    continuation_word_count
)


if continuation_word_count == 128:

    print(
        "\n✓ Exactly 128 words generated."
    )

else:

    print(
        "\n⚠ Output contains "
        f"{continuation_word_count} words."
    )

[output removed before publishing: this cell printed raw dataset / test text]


In [44]:
# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

OUTPUT_DIR = Path("/kaggle/working/phase1_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [45]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF")

os.environ["HF_TOKEN"] = HF_TOKEN

login(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


In [46]:
from huggingface_hub import hf_hub_download

print(f"Downloading {TEST_FILE} ...")

test_data_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

print("Path:", test_data_path)

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Loaded test examples:", len(test_data))

first_key = next(iter(test_data))
print("First ID:", first_key)
print("First prompt:")
print(test_data[first_key]["text"][:500])


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Loaded test examples: 1000
First ID: 1
First prompt:


In [47]:
# ============================================================
# RESUMABLE TEST GENERATION — SETUP
# ============================================================

import json
import time
from pathlib import Path


# ------------------------------------------------------------
# OUTPUT FILES
# ------------------------------------------------------------

PREDICTION_PROGRESS_PATH = Path(
    "/kaggle/working/prediction_progress.json"
)

FAILURE_LOG_PATH = Path(
    "/kaggle/working/prediction_failures.json"
)

FINAL_PREDICTION_PATH = Path(
    "/kaggle/working/prediction.json"
)


# ------------------------------------------------------------
# GENERATION SETTINGS
# ------------------------------------------------------------

GEN_MAX_NEW_TOKENS = 512
GEN_RETRY_MAX_NEW_TOKENS = 1024

GEN_TEMPERATURE = 0.7
GEN_TOP_K = 50
GEN_TOP_P = 0.95

GEN_REPETITION_PENALTY = 1.1   # 1.0 = off; try 1.1 for a separate submission

# Retry attempts get progressively more conservative sampling
# so a degenerate first sample is not repeated.
RETRY_TEMPERATURES = [GEN_TEMPERATURE, 0.7, 0.6]

MAX_RETRIES = 2

SAVE_EVERY_EXAMPLES = 10


# ------------------------------------------------------------
# LOAD PREVIOUS PROGRESS
# ------------------------------------------------------------

if False and PREDICTION_PROGRESS_PATH.exists():

    with open(
        PREDICTION_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        predictions = json.load(f)

else:

    predictions = {}


# ------------------------------------------------------------
# LOAD PREVIOUS FAILURES
# ------------------------------------------------------------

if FAILURE_LOG_PATH.exists():

    with open(
        FAILURE_LOG_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        failures = json.load(f)

else:

    failures = {}


# ------------------------------------------------------------
# STATUS
# ------------------------------------------------------------

total_test_examples = len(
    test_data
)

completed_examples = len(
    predictions
)

failed_examples = len(
    failures
)

remaining_examples = (
    total_test_examples
    - completed_examples
)


print("=" * 70)
print("TEST GENERATION STATUS")
print("=" * 70)

print(
    "Total test examples :",
    total_test_examples
)

print(
    "Already completed   :",
    completed_examples
)

print(
    "Previous failures   :",
    failed_examples
)

print(
    "Remaining           :",
    remaining_examples
)

print()
print(
    "✓ Generation setup ready."
)

TEST GENERATION STATUS
Total test examples : 1000
Already completed   : 0
Previous failures   : 0
Remaining           : 1000

✓ Generation setup ready.


In [48]:
# ============================================================
# RESUMABLE FULL TEST GENERATION  (FIXED)
# ============================================================
#
# Fixes vs. previous version:
#   1. The prompt is left-truncated per attempt so that
#      prompt + generation budget always fits in 2048 tokens.
#      (Before, long prompts silently capped generation, which
#      is why ID 411 could never reach 128 words.)
#   2. Early stopping at 128 words (much faster).
#   3. Retries use lower temperature.
#   4. No ID is ever left out: if every attempt is short, the
#      longest attempt is saved and logged in `short_outputs`.
#   5. IDs that previously "failed" are retried automatically.
# ============================================================

import json
import time
import re
from pathlib import Path


def count_words(text):
    return len(re.findall(r"\S+", text))


def first_n_words(text, n=128):
    return " ".join(re.findall(r"\S+", text)[:n])


def save_generation_state():
    for obj, path in (
        (predictions, PREDICTION_PROGRESS_PATH),
        (failures, FAILURE_LOG_PATH),
    ):
        tmp = path.with_suffix(".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        tmp.replace(path)


short_outputs = {}

model3.eval()
generation_start = time.time()
processed_since_save = 0
total_examples = len(test_data)

print("=" * 70)
print("STARTING OFFICIAL TEST GENERATION")
print("=" * 70)
print(f"Total examples     : {total_examples}")
print(f"Already completed  : {len(predictions)}")
print(f"To (re)try failed  : {len(failures)}")
print(f"Remaining          : {total_examples - len(predictions)}")


for index, (test_id, item) in enumerate(test_data.items(), start=1):

    test_id = str(test_id)

    if test_id in predictions:
        continue

    prompt = item.get("text", "")

    if not isinstance(prompt, str) or not prompt.strip():
        # Still must submit something for this ID
        predictions[test_id] = {"text": ""}
        failures[test_id] = {"error": "Empty prompt", "attempts": 0}
        processed_since_save += 1
        print(f"[{index}/{total_examples}] ID={test_id} ✗ empty prompt")
        continue

    prompt_ids = tokenizer.encode(prompt)

    best_text = ""
    best_words = -1
    done = False

    for attempt in range(1, MAX_RETRIES + 2):

        budget = (
            GEN_MAX_NEW_TOKENS if attempt == 1
            else GEN_RETRY_MAX_NEW_TOKENS
        )
        temperature = RETRY_TEMPERATURES[
            min(attempt - 1, len(RETRY_TEMPERATURES) - 1)
        ]

        # FIX: leave room for the generation budget
        max_prompt = CONFIG.max_seq_len - budget
        attempt_ids = prompt_ids[-max_prompt:]

        input_ids = torch.tensor(
            [attempt_ids], dtype=torch.long, device=device
        )

        try:
            generated_ids = generate(
                model=model3,
                input_ids=input_ids,
                max_new_tokens=budget,
                temperature=temperature,
                top_k=GEN_TOP_K,
                top_p=GEN_TOP_P,
                stop_words=128,
                repetition_penalty=GEN_REPETITION_PENALTY,
            )
        except Exception as e:
            print(f"[{index}/{total_examples}] ID={test_id} "
                  f"attempt={attempt} ⚠ {e!r}")
            continue

        new_token_ids = generated_ids[0, len(attempt_ids):].tolist()
        continuation = tokenizer.decode(new_token_ids)
        words = count_words(continuation)

        if words > best_words:
            best_words = words
            best_text = continuation

        if words >= 128:
            predictions[test_id] = {
                "text": first_n_words(continuation, 128)
            }
            
            failures.pop(test_id, None)
            done = True
            print(f"[{index}/{total_examples}] ID={test_id} "
                  f"✓ attempt={attempt} tokens={len(new_token_ids)}")
            break

        print(f"[{index}/{total_examples}] ID={test_id} "
              f"attempt={attempt} only {words} words, retrying")

    if not done:
        # FIX: never leave an ID missing — save the best attempt
        predictions[test_id] = {"text": first_n_words(best_text, 128)}
        failures.pop(test_id, None)
        short_outputs[test_id] = best_words
        print(f"    ⚠ saved short output ({best_words} words)")

    processed_since_save += 1

    if processed_since_save >= SAVE_EVERY_EXAMPLES:
        save_generation_state()
        mins = (time.time() - generation_start) / 60
        print(f"\n✓ Progress saved | done={len(predictions)}"
              f" | short={len(short_outputs)} | {mins:.1f} min\n")
        processed_since_save = 0


save_generation_state()

mins = (time.time() - generation_start) / 60
print()
print("=" * 70)
print("TEST GENERATION FINISHED")
print("=" * 70)
print("Total examples :", total_examples)
print("Predictions    :", len(predictions))
print("Short outputs  :", short_outputs if short_outputs else "none")
print("Time (minutes) :", f"{mins:.1f}")


STARTING OFFICIAL TEST GENERATION
Total examples     : 1000
Already completed  : 0
To (re)try failed  : 0
Remaining          : 1000
[1/1000] ID=1 ✓ attempt=1 tokens=288
[2/1000] ID=2 ✓ attempt=1 tokens=320
[3/1000] ID=3 ✓ attempt=1 tokens=288
[4/1000] ID=4 ✓ attempt=1 tokens=320
[5/1000] ID=5 ✓ attempt=1 tokens=288
[6/1000] ID=6 ✓ attempt=1 tokens=288
[7/1000] ID=7 ✓ attempt=1 tokens=320
[8/1000] ID=8 ✓ attempt=1 tokens=352
[9/1000] ID=9 ✓ attempt=1 tokens=320
[10/1000] ID=10 ✓ attempt=1 tokens=320

✓ Progress saved | done=10 | short=0 | 3.2 min

[11/1000] ID=11 ✓ attempt=1 tokens=320
[12/1000] ID=12 ✓ attempt=1 tokens=320
[13/1000] ID=13 ✓ attempt=1 tokens=320
[14/1000] ID=14 ✓ attempt=1 tokens=320
[15/1000] ID=15 ✓ attempt=1 tokens=320
[16/1000] ID=16 ✓ attempt=1 tokens=320
[17/1000] ID=17 ✓ attempt=1 tokens=320
[18/1000] ID=18 ✓ attempt=1 tokens=320
[19/1000] ID=19 ✓ attempt=1 tokens=320
[20/1000] ID=20 ✓ attempt=1 tokens=320

✓ Progress saved | done=20 | short=0 | 6.4 min

[21/1000

In [49]:
# ============================================================
# VALIDATE GENERATED PREDICTIONS  (FIXED)
# ============================================================

import json
import re

with open(PREDICTION_PROGRESS_PATH, "r", encoding="utf-8") as f:
    predictions = json.load(f)

test_ids = {str(k) for k in test_data.keys()}
prediction_ids = {str(k) for k in predictions.keys()}

missing_ids = test_ids - prediction_ids
unexpected_ids = prediction_ids - test_ids

print("=" * 70)
print("PREDICTION VALIDATION")
print("=" * 70)
print("Test examples :", len(test_ids))
print("Predictions   :", len(prediction_ids))
print("Missing       :", len(missing_ids))
print("Unexpected    :", len(unexpected_ids))

if unexpected_ids:
    raise ValueError(f"Unexpected IDs: {sorted(unexpected_ids)[:20]}")

errors = []
short = []

for test_id, item in predictions.items():
    if not isinstance(item, dict) or not isinstance(item.get("text"), str):
        errors.append((test_id, "bad structure"))
        continue
    n = len(re.findall(r"\S+", item["text"]))
    if n > 128:
        errors.append((test_id, f"{n} words (>128)"))
    elif n < 128:
        short.append((test_id, n))

if errors:
    for e in errors[:20]:
        print("  ✗", e)
    raise ValueError(f"{len(errors)} predictions are malformed.")

if short:
    print(f"\n⚠ {len(short)} predictions shorter than 128 words "
          "(allowed, but will score lower):")
    for tid, n in short[:20]:
        print(f"  ID {tid}: {n} words")

print()
if missing_ids:
    print("⚠ Not ready: generation still has missing IDs:",
          sorted(missing_ids, key=lambda x: int(x) if x.isdigit() else x)[:20])
else:
    print("✓ Every test ID has a prediction. Ready to package.")


[output removed before publishing: this cell printed raw dataset / test text]


In [50]:
# ============================================================
# CREATE FINAL prediction.json + submission.zip  (FIXED)
# ============================================================

import json
import zipfile
from pathlib import Path

test_ids_ordered = [str(k) for k in test_data.keys()]
missing_ids = [t for t in test_ids_ordered if t not in predictions]

if missing_ids:
    raise RuntimeError(
        f"{len(missing_ids)} test IDs have no prediction. "
        "Re-run the generation cell first."
    )

final_predictions = {
    tid: {"text": predictions[tid]["text"]}
    for tid in test_ids_ordered
}

FINAL_PREDICTION_PATH = Path("/kaggle/working/prediction.json")
tmp = FINAL_PREDICTION_PATH.with_suffix(".tmp")
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(final_predictions, f, ensure_ascii=False, indent=2)
tmp.replace(FINAL_PREDICTION_PATH)

# prediction.json must sit at the ROOT of the zip
SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(FINAL_PREDICTION_PATH, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
assert names == ["prediction.json"], names

print("=" * 70)
print("SUBMISSION READY")
print("=" * 70)
print("prediction.json :", FINAL_PREDICTION_PATH,
      f"({FINAL_PREDICTION_PATH.stat().st_size / 1024:.1f} KB)")
print("submission.zip  :", SUBMISSION_ZIP, "->", names)
print("Examples        :", len(final_predictions))
print()
print("First prediction:")
print(final_predictions[test_ids_ordered[0]]["text"])


SUBMISSION READY
prediction.json : /kaggle/working/prediction.json (717.8 KB)
submission.zip  : /kaggle/working/submission.zip -> ['prediction.json']
Examples        : 1000

First prediction:
